# Noiseless least-norm inversion with `pygeoinf3D`

This notebook shows how to use `pygeoinf`'s existing minimum-norm inversion workflow with a concrete 3D forward operator from `pygeoinf3D`. The model space remains basis-free: the solution is a `Function3D`, not a coefficient vector.

For consistent exact data and linearly independent measurement functionals, the minimum-norm solution is

$$\hat u = G^*(G G^*)^{-1} d.$$

More generally, the robust data-side formula uses the Moore-Penrose pseudoinverse,

$$\hat u = G^*(G G^*)^{\dagger} d,$$

which is what we compute below with an eigen-based solver. Here $G$ is a `SOLAOperator3D` built from Gaussian volumetric kernels on a box domain.

In [ ]:
import numpy as np
from pygeoinf import (
    EuclideanSpace,
    LinearForwardProblem,
    LinearMinimumNormInversion,
    EigenSolver,
)
from pygeoinf3d import (
    BoxDomain,
    Lebesgue3D,
    Function3D,
    FunctionToKernelAdapter,
    SOLAOperator3D,
)
from pygeoinf3d.providers.function_providers import GaussianProvider3D
from pygeoinf3d.plot import compare_function3d_axis_sliders

np.random.seed(7)


## Build a 3D SOLA forward operator

We place several tens of Gaussian volumetric kernels in a unit cube and use them to define a finite-dimensional measurement operator.

In [ ]:
box = BoxDomain(np.zeros(3), np.ones(3))
space = Lebesgue3D(box)

n_kernels = 100
centers = np.random.uniform(0.15, 0.85, size=(n_kernels, 3))
sigmas = np.random.uniform(0.08, 0.16, size=(n_kernels, 3))

function_provider = GaussianProvider3D(
    box,
    centers=centers,
    sigmas=sigmas,
    normalized=True,
)
kernel_provider = FunctionToKernelAdapter(function_provider, space)
G = SOLAOperator3D(space, EuclideanSpace(n_kernels), kernel_provider)

print(f"Number of kernels: {n_kernels}")
print(f"Forward operator domain: {type(G.domain).__name__}")
print(f"Forward operator codomain dimension: {G.codomain.dim}")

## Define a smooth ground-truth model and generate noiseless data

The callable must handle both a single point of shape `(3,)` and a batch of points of shape `(n, 3)`.

In [ ]:
def true_model_callable(x):
    x = np.asarray(x, dtype=float)
    single_point = x.ndim == 1
    x2 = x[None, :] if single_point else x

    c1 = np.array([0.30, 0.35, 0.55])
    c2 = np.array([0.72, 0.62, 0.38])
    c3 = np.array([0.45, 0.75, 0.72])

    blob1 = 1.3 * np.exp(-np.sum((x2 - c1) ** 2, axis=1) / (2.0 * 0.09 ** 2))
    blob2 = -0.9 * np.exp(-np.sum((x2 - c2) ** 2, axis=1) / (2.0 * 0.12 ** 2))
    blob3 = 0.7 * np.exp(-np.sum((x2 - c3) ** 2, axis=1) / (2.0 * 0.07 ** 2))
    trend = 0.15 * (x2[:, 0] - 0.5) - 0.10 * (x2[:, 1] - 0.5)
    values = blob1 + blob2 + blob3 + trend
    return float(values[0]) if single_point else values

u_true = Function3D(box, evaluate_callable=true_model_callable, name="true model")
d = G(u_true)

print("Data shape:", d.shape)
print("First five data values:", np.round(d[:5], 6))

## Solve the minimum-norm inverse problem with pygeoinf



For zero data error, `LinearMinimumNormInversion` uses the data-space operator `G G^*` and applies `G^*` afterwards. When `G G^*` is strictly invertible this is exactly `G^*(G G^*)^{-1}`; for dependent or nearly dependent kernels, an eigen-based pseudoinverse is the safer interpretation and is what we use here. The model space still stays basis-free because all finite-dimensional linear algebra happens on the Euclidean data side.



This eigen-based solve is appropriate here because the demo uses only a few dozen kernels. For much larger kernel families, the data-side linear algebra becomes the main scaling bottleneck and iterative or regularized variants are more appropriate.

In [ ]:
forward_problem = LinearForwardProblem(G)
inversion = LinearMinimumNormInversion(forward_problem)
inverse_operator = inversion.minimum_norm_operator(
    EigenSolver(galerkin=True, rtol=1.0e-12, parallel=True)
)
u_hat = inverse_operator(d)

residual = np.linalg.norm(G(u_hat) - d)
print(f"Residual norm ||G u_hat - d|| = {residual:.3e}")
print("Recovered object type:", type(u_hat).__name__)


## Visualize the recovered model

The interactive Plotly slider lets you move a slice plane through the box along `x`, `y`, or `z` and compare the true model, least-norm solution, and absolute total sensitivity side by side with a shared slider.

In [ ]:
def _total_sensitivity_callable(x):
    x = np.asarray(x)
    total = sum(kernel_provider.get_kernel(i).kernel(x) for i in range(n_kernels))
    return np.abs(total)

total_sensitivity = Function3D(
    box,
    evaluate_callable=_total_sensitivity_callable,
    name="Abs. total sensitivity",
)

try:
    comparison_fig = compare_function3d_axis_sliders(
        [u_true, u_hat, total_sensitivity],
        axis="z",
        n_slices=20,
        resolution=60,
        show_plot=False,
        titles=["True model", "Least-norm solution", "Abs. total sensitivity"],
        colorscale="RdBu_r",
        width=900,
        height=400,
    )
    comparison_fig.show()
except ImportError as exc:
    print(
        "Interactive comparison unavailable. Install pygeoinf3d[interactive] "
        f"to enable Plotly-based slicing. ({exc})"
    )


## Remarks

- The inversion used pygeoinf's standard `LinearMinimumNormInversion` workflow.
- The model space remained basis-free throughout: no `to_components` or `from_components` calls were needed on `Lebesgue3D`.
- All finite-dimensional linear algebra happened on the data side through the operator `G G^*`.
- With more kernels, the same workflow applies, but the conditioning of `G G^*` becomes the main numerical issue to monitor.